# Exploratory Data Analysis & Feature Engineering Comparison

In this notebook, we compare the correlation patterns in the Maternal Health Risk dataset before and after applying clinical feature engineering.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# Set Seaborn theme for premium aesthetics
sns.set_theme(style="whitegrid", palette="magma")

# Load the real dataset
data_path = os.path.join("..", "data", "Maternal Health Risk Data Set.csv")
df = pd.read_csv(data_path)

# Encode RiskLevel for correlation analysis
risk_map = {'low risk': 0, 'mid risk': 1, 'high risk': 2}
df_encoded = df.copy()
df_encoded['RiskLevel'] = df_encoded['RiskLevel'].map(risk_map)

print("Data loaded and target encoded.")
df.head()

## 1. Class Distribution

Using Seaborn's `countplot` to visualize the frequency of each risk category.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='RiskLevel', order=['low risk', 'mid risk', 'high risk'])
plt.title("Distribution of Maternal Health Risk Levels", fontsize=15)
plt.ylabel("Count (Patients)")
plt.show()

## 2. Feature Distribution by Risk Level

Analyzing how key physiological metrics vary across different risk categories using Seaborn `boxplots`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.boxplot(ax=axes[0], data=df, x='RiskLevel', y='BS', order=['low risk', 'mid risk', 'high risk'])
axes[0].set_title("Blood Sugar vs Risk Level")

sns.boxplot(ax=axes[1], data=df, x='RiskLevel', y='SystolicBP', order=['low risk', 'mid risk', 'high risk'])
axes[1].set_title("Systolic BP vs Risk Level")

sns.boxplot(ax=axes[2], data=df, x='RiskLevel', y='Age', order=['low risk', 'mid risk', 'high risk'])
axes[2].set_title("Age vs Risk Level")

plt.tight_layout()
plt.show()

## 3. Correlation Matrix: BEFORE Feature Engineering

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix (Baseline Features)", fontsize=15)
plt.show()

## 4. Applying Feature Engineering

We apply clinical metrics such as Mean Arterial Pressure (MAP), Pulse Pressure, and Shock Index.

In [ ]:
# Clinical Feature Engineering
df_encoded["MAP"] = (df_encoded["SystolicBP"] + 2 * df_encoded["DiastolicBP"]) / 3
df_encoded["PulsePressure"] = df_encoded["SystolicBP"] - df_encoded["DiastolicBP"]
df_encoded["ShockIndex"] = df_encoded["HeartRate"] / df_encoded["SystolicBP"]
df_encoded["BPRatio"] = df_encoded["SystolicBP"] / df_encoded["DiastolicBP"]

df_encoded["TempDeviation"] = df_encoded["BodyTemp"] - 98.2
df_encoded["BSDeviation"] = df_encoded["BS"] - 7

df_encoded["CombinedRiskScore"] = (
    (df_encoded["MAP"] > 105).astype(int) +
    (df_encoded["BS"] > 10).astype(int) +
    (df_encoded["HeartRate"] > 90).astype(int) +
    (df_encoded["TempDeviation"] > 1).astype(int)
)

print("Feature engineering complete.")

## 5. Correlation Matrix: AFTER Feature Engineering

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix (Engineered Features)", fontsize=15)
plt.show()

## 6. Model Evaluation: Confusion Matrices

To fully understand our model's performance, we load the trained models from the pipeline and evaluate them on our test data. A Confusion Matrix shows us exactly where the model is predicting correctly, and where it is making false positive/negative errors.

In [ ]:
import joblib
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure we can import our src modules
sys.path.append(os.path.abspath(os.path.join("..", "src")))
from data_preprocessing import prepare_data_for_modeling

# Re-load and split the data (using same random state as pipeline)
X_train, X_test, y_train, y_test, le = prepare_data_for_modeling(df_encoded, use_smote=False)

# The models we trained in our pipeline
model_files = [
    "Decision_Tree.pkl",
    "Random_Forest.pkl",
    "Gradient_Boosting.pkl",
    "XGBoost.pkl",
    "Voting_Ensemble.pkl"
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, m_file in enumerate(model_files):
    # Load model
    model_path = os.path.join("..", "models", m_file)
    if not os.path.exists(model_path):
        continue
        
    model = joblib.load(model_path)
    
    # Predict
    preds = model.predict(X_test)
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['high risk', 'low risk', 'mid risk'])
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d', colorbar=False)
    axes[idx].set_title(f"{m_file.replace('.pkl', '').replace('_', ' ')}")
    axes[idx].grid(False)

# Hide the last empty subplot
if len(model_files) < len(axes):
    for i in range(len(model_files), len(axes)):
        fig.delaxes(axes[i])

plt.tight_layout()
plt.show()